<div style="background: linear-gradient(135deg, #e83e8c 0%, #6f42c1 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">⛰️ 02 — Gradiente e Regra da Cadeia</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 01 · Bloco 06 · Cálculo</p>
</div>

## 🎯 Objetivos deste notebook

Ao finalizar este notebook você vai:
- Entender gradiente como **vetor de derivadas parciais**
- Implementar gradiente numérico e analítico
- Dominar a Regra da Cadeia (fundamento do backpropagation)
- Visualizar o campo gradiente de funções 2D

---

## 1️⃣ Derivadas Parciais: Uma Dimensão por Vez

Se `f(x, y) = x² + 2xy + y²`, a derivada parcial em relação a `x` é:
- `∂f/∂x = 2x + 2y` (trate `y` como constante)
- `∂f/∂y = 2x + 2y` (trate `x` como constante)

O **gradiente** é o vetor: `∇f = [∂f/∂x, ∂f/∂y]`

Ele aponta na **direção de maior crescimento** da função.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Função: f(x, y) = x^2 + y^2 (paraboloide)
def f(x, y):
    return x**2 + y**2

# Gradiente analítico
def grad_f(x, y):
    return np.array([2*x, 2*y])  # [∂f/∂x, ∂f/∂y]

# Gradiente numérico (aproximação por diferença finita)
def grad_numerico(func, x, y, h=1e-5):
    df_dx = (func(x+h, y) - func(x-h, y)) / (2*h)
    df_dy = (func(x, y+h) - func(x, y-h)) / (2*h)
    return np.array([df_dx, df_dy])

# Testar em (3, 4)
ponto = (3, 4)
print(f'Ponto: {ponto}')
print(f'Gradiente analítico:  {grad_f(*ponto)}')
print(f'Gradiente numérico:   {grad_numerico(f, *ponto).round(6)}')
print(f'Diferença: {np.abs(grad_f(*ponto) - grad_numerico(f, *ponto)).max():.2e}')

## 2️⃣ Visualizando o Campo Gradiente

Vamos ver para onde o gradiente aponta em vários pontos do espaço:

In [ ]:
# Criar grade de pontos
x_range = np.linspace(-3, 3, 15)
y_range = np.linspace(-3, 3, 15)
X, Y = np.meshgrid(x_range, y_range)
Z = f(X, Y)

# Calcular gradiente em cada ponto
GX = 2 * X  # ∂f/∂x
GY = 2 * Y  # ∂f/∂y

fig, ax = plt.subplots(figsize=(10, 8))
contour = ax.contourf(X, Y, Z, levels=20, cmap='coolwarm', alpha=0.6)
plt.colorbar(contour, label='f(x,y)')
ax.quiver(X, Y, -GX, -GY, color='black', alpha=0.7)  # Negativo = direção de descida
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Campo Gradiente de f(x,y) = x² + y²\nSetas = direção de DESCIDA', 
             fontsize=14, fontweight='bold')
ax.set_aspect('equal')
plt.show()

## 3️⃣ A Regra da Cadeia — O Coração do Backpropagation

Se temos funções compostas `f(g(x))`, a derivada é:

```
d/dx f(g(x)) = f'(g(x)) · g'(x)
```

**Por que importa em IA?** Uma rede neural é uma composição gigante:
```
saída = f₃(f₂(f₁(x · w₁) · w₂) · w₃)
```
Para atualizar `w₁`, precisamos "propagar o erro para trás" usando a regra da cadeia.

In [ ]:
# Exemplo: f(g(x)) onde g(x) = x² e f(u) = sin(u)
# Então f(g(x)) = sin(x²)
# Derivada pela regra da cadeia: cos(x²) · 2x

def g(x): return x**2
def f_composta(x): return np.sin(g(x))

# Derivada analítica pela regra da cadeia
def derivada_analitica(x):
    return np.cos(x**2) * 2*x  # f'(g(x)) · g'(x)

# Derivada numérica para validar
def derivada_numerica(func, x, h=1e-7):
    return (func(x+h) - func(x-h)) / (2*h)

x_test = np.array([0.5, 1.0, 2.0, 3.0])
print('x      | Analítica | Numérica  | Erro')
print('-' * 50)
for x in x_test:
    ana = derivada_analitica(x)
    num = derivada_numerica(f_composta, x)
    print(f'{x:<6.1f} | {ana:>9.6f} | {num:>9.6f} | {abs(ana-num):.2e}')

## 4️⃣ Regra da Cadeia com 3 Funções (Rede Neural Simulada)

Simulemos uma "rede" de 3 camadas: `h = relu(x·w)`, `out = sigmoid(h)`, `loss = (out - y)²`

In [ ]:
# Simulação de forward + backward para 1 neurônio

# Funções
def relu(x): return np.maximum(0, x)
def sigmoid(x): return 1 / (1 + np.exp(-x))

# Derivadas
def relu_deriv(x): return (x > 0).astype(float)
def sigmoid_deriv(x): 
    s = sigmoid(x)
    return s * (1 - s)

# Forward pass
x = 2.0
w = 0.5
y_true = 1.0

z = x * w              # Camada 1: linear
h = relu(z)            # Camada 2: ativação
out = sigmoid(h)       # Camada 3: saída
loss = (out - y_true)**2  # Perda

print(f'Forward: x={x}, w={w}')
print(f'  z = x·w = {z}')
print(f'  h = relu(z) = {h}')
print(f'  out = sigmoid(h) = {out:.6f}')
print(f'  loss = (out - y)² = {loss:.6f}')

# Backward pass (regra da cadeia)
dL_dout = 2 * (out - y_true)           # ∂loss/∂out
dout_dh = sigmoid_deriv(h)              # ∂out/∂h
dh_dz = relu_deriv(z)                   # ∂h/∂z
dz_dw = x                               # ∂z/∂w

# Regra da cadeia: ∂loss/∂w = ∂loss/∂out · ∂out/∂h · ∂h/∂z · ∂z/∂w
dL_dw = dL_dout * dout_dh * dh_dz * dz_dw

print(f'\nBackward (regra da cadeia):')
print(f'  ∂loss/∂out = {dL_dout:.6f}')
print(f'  ∂out/∂h    = {dout_dh:.6f}')
print(f'  ∂h/∂z      = {dh_dz:.6f}')
print(f'  ∂z/∂w      = {dz_dw:.6f}')
print(f'  ∂loss/∂w   = {dL_dw:.6f}')

# Atualizar peso
lr = 0.1
w_novo = w - lr * dL_dw
print(f'\nw atualizado: {w:.4f} → {w_novo:.4f}')

## 🏋️ Questões para Praticar

### Q1. Gradiente de Rosenbrock
Implemente `f(x,y) = (1-x)² + 100(y-x²)²` (função de Rosenbrock). Calcule o gradiente analiticamente e valide numericamente no ponto (1.5, 2.0).

### Q2. Campo Gradiente
Plote o campo gradiente da função `f(x,y) = sin(x) · cos(y)`. Use `quiver`.

### Q3. Loop de Treino
Usando o neurônio simulado acima, faça um loop de 100 iterações atualizando `w`. Plote a loss convergindo ao longo das iterações.

### Q4. Regra da Cadeia Tripla
Dada `f(x) = ln(sin(x²))`, calcule a derivada pela regra da cadeia (são 3 funções compostas). Valide numericamente.

In [ ]:
# ============================================
# Resolva as questões aqui
# ============================================

# Q1: Rosenbrock


# Q2: Campo gradiente


# Q3: Loop de treino


# Q4: Regra da cadeia tripla
